In [4]:
import os
import warnings
import requests
import librosa
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column
from bokeh.models import Span, LinearColorMapper, ColorBar, FixedTicker

# Initialize Bokeh for notebook output
output_notebook()

# --- Utility Functions (Unchanged) ---
def download_audio(audio_url: str, local_path: str):
    if not os.path.exists(local_path):
        print(f"Downloading audio from {audio_url}...")
        response = requests.get(audio_url)
        response.raise_for_status()
        with open(local_path, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded audio to {local_path}.")
    else:
        print(f"Audio file {local_path} already exists. Skipping download.")

def load_cuepoints(url: str, name: str, color: str, time_column_name: str = 'time'):
    """Loads cue points from a CSV file using a specified column name."""
    try:
        df = pd.read_csv(url)
        peaks_in_sec = df[time_column_name].values
        return {'name': name, 'times': peaks_in_sec, 'color': color}
    except KeyError:
        warnings.warn(f"Failed to load cuepoints from {url}: Column '{time_column_name}' not found.")
        return None
    except Exception as e:
        warnings.warn(f"Failed to load cuepoints from {url}: {e}")
        return None

# FIX: This function now calculates two distinct features for envelope and flux.
def extract_all_features(y: np.ndarray, sr: int, hop_length: int, n_fft_stft: int, hop_length_stft: int):
    """Extract all necessary audio features from the full audio signal at once."""
    print("Extracting features for the entire audio file... (this may take a moment)")
    features = {}
    
    # --- Features with common hop_length ---
    
    # FIX: Calculate true Amplitude Envelope using Root-Mean-Square (RMS) energy.
    # This measures the loudness/power in each frame.
    features['amplitude_envelope'] = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    
    # Spectral Flux is a measure of how quickly the power spectrum is changing.
    # librosa.onset.onset_strength is the correct function for this.
    features['spectral_flux'] = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length, aggregate=np.mean)
    
    # Other features remain the same
    features['zero_crossing_rate'] = librosa.feature.zero_crossing_rate(y=y, hop_length=hop_length)[0]
    features['spectral_centroid'] = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)[0]
    
    S = librosa.feature.melspectrogram(y=y, sr=sr, hop_length=hop_length)
    features['melspectrogram_db'] = librosa.power_to_db(S, ref=np.max)
    
    # --- STFT-based features (can have different hop_length) ---
    stft = librosa.stft(y=y, n_fft=n_fft_stft, hop_length=hop_length_stft)
    features['stft_db'] = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
    
    print("Feature extraction complete.")
    return features

# --- Plotting Functions (Unchanged) ---
def plot_feature(feature_data: np.ndarray, times: np.ndarray, title: str, y_label: str, width=900, height=250):
    p = figure(title=title, x_axis_label='Time (s)', y_axis_label=y_label, width=width, height=height)
    p.line(times, feature_data, line_width=2)
    return p

def plot_spectrogram(spec_db: np.ndarray, times: np.ndarray, sr: int, title: str, y_label: str, n_mels: int = 128, width=900, height=400):
    p = figure(title=title, x_axis_label='Time (s)', y_axis_label=y_label, 
               width=width, height=height, x_range=(times[0], times[-1]))
    color_mapper = LinearColorMapper(palette="Viridis256", low=np.min(spec_db), high=np.max(spec_db))
    if "Mel" in y_label:
        dh = n_mels
        mel_frequencies = librosa.mel_frequencies(n_mels=n_mels, fmin=0, fmax=sr/2)
        tick_positions = np.linspace(0, n_mels - 1, num=10, dtype=int)
        tick_labels = {int(pos): f"{mel_frequencies[pos]:.0f} Hz" for pos in tick_positions}
        p.yaxis.ticker = FixedTicker(ticks=tick_positions.tolist())
        p.yaxis.major_label_overrides = tick_labels
    else:
        dh = sr / 2
    p.image(image=[spec_db], x=times[0], y=0, dw=times[-1] - times[0], dh=dh, color_mapper=color_mapper)
    color_bar = ColorBar(color_mapper=color_mapper, label_standoff=12, location=(0, 0), title='dB')
    p.add_layout(color_bar, 'right')
    return p

def add_lines_to_plots(plots: dict, lines_to_plot: list, cuepoint_lists: list, custom_intervals: list, start_time: float, end_time: float):
    for plot in plots.values():
        if 'custom_interval_lines' in lines_to_plot:
            for interval_set in custom_intervals:
                start, step, color = interval_set['start_time'], interval_set['interval'], interval_set.get('color', 'red')
                if start >= start_time: current_time = start
                else: current_time = start + np.ceil((start_time - start) / step) * step
                while current_time <= end_time:
                    plot.add_layout(Span(location=current_time, dimension='height', line_color=color, line_dash='dotted', line_width=2))
                    current_time += step
        if 'cuepoints' in lines_to_plot:
            for cuepoint in cuepoint_lists:
                relevant_times = cuepoint['times'][(cuepoint['times'] >= start_time) & (cuepoint['times'] <= end_time)]
                for cue_time in relevant_times:
                    plot.add_layout(Span(location=cue_time, dimension='height', line_color=cuepoint['color'], line_dash='dashed', line_width=2))

# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
local_audio_path = 'examples/Dufour_full_mono.aif'
start_times = [0, 30, 60]  # List of start times in seconds
end_times = [4, 34, 64]    # List of end times in seconds
features_to_plot = [
    'amplitude_envelope', 'spectral_flux',
    'melspectrogram'
] # Options: 'amplitude_envelope', 'spectral_flux', 'zero_crossing_rate', 'spectral_centroid', 'melspectrogram', 'stft_spectrogram'

lines_to_plot = ['custom_interval_lines'] # Options: 'custom_interval_lines', 'cuepoints'
custom_intervals = [
    {'start_time': 0.398730159, 'interval': 2.234, 'color': 'red'},
]
HOP_LENGTH, N_MELS = 512, 128
N_FFT_STFT, HOP_LENGTH_STFT = 2048, 512

# ==============================================================================
# --- MAIN SCRIPT ---
# ==============================================================================
y, sr = librosa.load(local_audio_path, sr=None)
duration = librosa.get_duration(y=y, sr=sr)
print(f"The audio file is {duration:.2f} seconds long with a sample rate of {sr} Hz.")

cuepoint_urls = [
    {
        'url': 'https://raw.githubusercontent.com/egorpol/beat_it/refs/heads/main/csv/dufour_manual.csv', 
        'name': 'Onsets Manual', 
        'color': 'cyan',
        'time_column_name': 'peaks_in_sec' # Specify the column name here
    }
]
cuepoint_lists = [cp for cp in (load_cuepoints(**c) for c in cuepoint_urls) if cp]

all_features = extract_all_features(y, sr, HOP_LENGTH, N_FFT_STFT, HOP_LENGTH_STFT)
times_common = librosa.frames_to_time(np.arange(all_features['amplitude_envelope'].shape[0]), sr=sr, hop_length=HOP_LENGTH)
times_stft = librosa.frames_to_time(np.arange(all_features['stft_db'].shape[1]), sr=sr, hop_length=HOP_LENGTH_STFT)

segment_layouts = []
for idx, (start_time, end_time) in enumerate(tqdm(zip(start_times, end_times), total=len(start_times), desc="Processing Segments"), 1):
    if start_time < 0 or end_time > duration:
        warnings.warn(f"Segment {idx}: Time range is out of bounds. Skipping.")
        continue

    plots = {}
    start_frame_common = np.searchsorted(times_common, start_time)
    end_frame_common = np.searchsorted(times_common, end_time)
    start_frame_stft = np.searchsorted(times_stft, start_time)
    end_frame_stft = np.searchsorted(times_stft, end_time)
    seg_times_common = times_common[start_frame_common:end_frame_common]
    seg_times_stft = times_stft[start_frame_stft:end_frame_stft]

    # Plotting logic remains the same, but will now use the corrected data
    if 'amplitude_envelope' in features_to_plot:
        data = all_features['amplitude_envelope'][start_frame_common:end_frame_common]
        plots['amp'] = plot_feature(data, seg_times_common, f"Amplitude Envelope (Seg {idx})", 'RMS Energy')
    if 'spectral_flux' in features_to_plot:
        data = all_features['spectral_flux'][start_frame_common:end_frame_common]
        plots['flux'] = plot_feature(data, seg_times_common, f"Spectral Flux (Seg {idx})", 'Flux')
    if 'zero_crossing_rate' in features_to_plot:
        data = all_features['zero_crossing_rate'][start_frame_common:end_frame_common]
        plots['zcr'] = plot_feature(data, seg_times_common, f"Zero-Crossing Rate (Seg {idx})", 'ZCR')
    if 'spectral_centroid' in features_to_plot:
        data = all_features['spectral_centroid'][start_frame_common:end_frame_common]
        plots['centroid'] = plot_feature(data, seg_times_common, f"Spectral Centroid (Seg {idx})", 'Centroid (Hz)')
    if 'melspectrogram' in features_to_plot:
        data = all_features['melspectrogram_db'][:, start_frame_common:end_frame_common]
        plots['mel'] = plot_spectrogram(data, seg_times_common, sr, f"Mel-Spectrogram (Seg {idx})", 'Frequency (Mel)', n_mels=N_MELS)
    if 'stft_spectrogram' in features_to_plot:
        data = all_features['stft_db'][:, start_frame_stft:end_frame_stft]
        plots['stft'] = plot_spectrogram(data, seg_times_stft, sr, f"STFT Spectrogram (Seg {idx})", 'Frequency (Hz)')

    add_lines_to_plots(plots, lines_to_plot, cuepoint_lists, custom_intervals, start_time, end_time)
    segment_layouts.append(column(list(plots.values())))

if segment_layouts:
    layout = column(*segment_layouts)
    show(layout)
else:
    print("No segments were plotted.")

Loading BokehJS ...

The audio file is 77.67 seconds long with a sample rate of 44100 Hz.
Extracting features for the entire audio file... (this may take a moment)
Feature extraction complete.


Processing Segments:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
def save_projected_lines_to_csv(custom_intervals: list, duration: float, output_path: str, save_separately: bool = False):
    """
    Generates time cues from custom interval definitions and saves them to CSV files.

    Args:
        custom_intervals (list): A list of dictionaries, each defining a set of 
                                 repeating lines with 'start_time', 'interval', and 'color'.
        duration (float): The total duration of the audio file in seconds.
        output_path (str): The file path for the output. The behavior depends on save_separately.
        save_separately (bool): 
            - If False (default), all cue points are merged, sorted, and saved to a 
              single file at `output_path`.
            - If True, each interval set is saved to its own file. The filename is
              derived from `output_path` by appending the interval's color 
              (e.g., 'output_red.csv', 'output_orange.csv').
    """
    if not save_separately:
        # Original behavior: merge all into one file
        print("Saving all projected cue points into a single file...")
        all_cue_points = []
        for interval_set in custom_intervals:
            current_time = interval_set['start_time']
            interval = interval_set['interval']
            while current_time <= duration:
                all_cue_points.append(current_time)
                current_time += interval
        
        if not all_cue_points:
            print("No cue points were generated. CSV file will not be created.")
            return

        df = pd.DataFrame(sorted(all_cue_points), columns=['onset_times'])
        df.to_csv(output_path, index=False, float_format='%.6f')
        print(f"Successfully saved {len(all_cue_points)} combined cue points to {output_path}")

    else:
        # New behavior: save one file per interval layer
        print("Saving projected cue points into separate files for each layer...")
        base_path, extension = os.path.splitext(output_path)
        
        for i, interval_set in enumerate(custom_intervals):
            cue_points_for_layer = []
            current_time = interval_set['start_time']
            interval = interval_set['interval']
            
            while current_time <= duration:
                cue_points_for_layer.append(current_time)
                current_time += interval

            if not cue_points_for_layer:
                warnings.warn(f"No cue points generated for layer {i+1}. Skipping file creation.")
                continue

            # Use the color for a descriptive filename, with a fallback
            layer_name = interval_set.get('color', f'layer_{i+1}')
            separate_output_path = f"{base_path}_{layer_name}{extension}"
            
            # No need to sort here as cues are generated chronologically for a single layer
            df = pd.DataFrame(cue_points_for_layer, columns=['onset_times'])
            df.to_csv(separate_output_path, index=False, float_format='%.6f')
            print(f"Successfully saved {len(cue_points_for_layer)} cue points for layer '{layer_name}' to {separate_output_path}")

output_csv_path = 'examples/projected_cue_points_dufour.csv' 
save_projected_lines_to_csv(custom_intervals, duration, output_csv_path, save_separately=True)

Saving projected cue points into separate files for each layer...
Successfully saved 35 cue points for layer 'red' to examples/projected_cue_points_dufour_red.csv
